# CFP_LagLlama_Forecasts

Generates 1-step-ahead VaR and ES forecasts using Lag-Llama (7.5M) via Student-t distribution head with lag features. 1000 samples per day; extracts VaR quantiles, empirical ES at 2.5% (FRTB), and fitted Student-t parameters. Seed: 42.

**Paper:** Pele, Lessmann, Hardle (2026)

In [1]:
# Install dependencies (uncomment as needed)

!pip install einops gluonts==0.14.4 uni2ts huggingface_hub lightning  #" Moirai + LagLlama"

#!pip install torch --index-url https://download.pytorch.org/whl/cu124  # GPU
#!pip install chronos-forecasting yfinance pandas numpy scipy tqdm pyarrow openpyxl arch

!pip install git+https://github.com/google-research/timesfm.git         # TimesFM
!git clone https://github.com/time-series-foundation-models/lag-llama.git  # LagLlama

  Cloning https://github.com/google-research/timesfm.git to /tmp/pip-req-build-_d57xsht
  Running command git clone --filter=blob:none --quiet https://github.com/google-research/timesfm.git /tmp/pip-req-build-_d57xsht
  Resolved https://github.com/google-research/timesfm.git to commit d720daa6786539c2566a44464fbda1019c0a82c0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
fatal: destination path 'lag-llama' already exists and is not an empty directory.


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

DATA_DIR = Path('cfp_ijf_data')
RET_DIR = DATA_DIR / 'returns'

ASSETS = [
    'SP500','STOXX','GDAXI','FCHI','FTSE100','ICLN','NIKKEI','HSI','BOVESPA','NIFTY','ASX200',
    'CBU0','TLT','IBGL','DJCI','GOLD','WTI','NATGAS','BTC','ETH','EURUSD','GBPUSD','USDJPY','AUDUSD'
]
ALPHAS = [0.01, 0.025, 0.05, 0.10]
CONTEXT = 512
N_SAMPLES = 1000

def load_returns(asset):
    df = pd.read_csv(RET_DIR / f'{asset}.csv', parse_dates=['date']).set_index('date').sort_index()
    df = df[df['log_return'].abs() <= 0.50]
    return df['log_return']

print(f'Assets: {len(ASSETS)}, Context: {CONTEXT}, Samples: {N_SAMPLES}')

MODELS = [('lagllama', 'time-series-foundation-models/Lag-Llama', 'Lag-Llama')]

Assets: 24, Context: 512, Samples: 1000


In [ ]:
import torch
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from tqdm import tqdm
import sys

sys.path.insert(0, "./lag-llama")
from lag_llama.gluon.estimator import LagLlamaEstimator
from gluonts.dataset.common import ListDataset

assert torch.cuda.is_available(), "CUDA not available"
device = torch.device("cuda")
print(f"Device: {device}")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

ckpt = hf_hub_download(
    repo_id="time-series-foundation-models/Lag-Llama",
    filename="lag-llama.ckpt"
)

estimator = LagLlamaEstimator(
    prediction_length=1,
    context_length=CONTEXT,
    input_size=1,
    n_layer=8,
    n_embd_per_head=36,
    n_head=4,
    num_parallel_samples=N_SAMPLES,
    batch_size=1,
    device=device,
    rope_scaling={"type": "linear", "factor": max(1.0, CONTEXT / 32)},
    ckpt_path=ckpt,
    time_feat=True,
)

transformation = estimator.create_transformation()
module = estimator.create_lightning_module()
predictor_ll = estimator.create_predictor(transformation, module)

out_dir = DATA_DIR / "lagllama"
out_dir.mkdir(parents=True, exist_ok=True)
print(f"Output: {out_dir}")

label = "Lag-Llama"
pbar = tqdm(ASSETS, desc=label)
for asset in pbar:
    pbar.set_description(f"{label} → {asset}")
    
    ret = load_returns(asset)
    n = len(ret)
    vals = ret.values
    dates = ret.index
    
    records = []
    
    for t in range(CONTEXT, n):
        ds = ListDataset(
            [{
                "target": vals[t-CONTEXT:t],
                "start": pd.Period(dates[t-CONTEXT], freq="D"),
            }],
            freq="D"
        )
        
        with torch.no_grad():
            forecasts = list(predictor_ll.predict(ds, num_samples=N_SAMPLES))
        
        samples = forecasts[0].samples.flatten()
        
        row = {"date": dates[t]}
        for alpha in ALPHAS:
            row[f"VaR_{alpha}"] = np.percentile(samples, alpha * 100)
        row["mean"] = samples.mean()
        row["std"] = samples.std()
        
        sorted_s = np.sort(samples)
        row["ES_empirical_0.025"] = sorted_s[:25].mean()
        row["ES_empirical_0.01"] = sorted_s[:10].mean()
        row["ES_empirical_0.05"] = sorted_s[:50].mean()
        
        records.append(row)
    
    df_out = pd.DataFrame(records).set_index("date")
    df_out.to_parquet(out_dir / f"{asset}.parquet")

print(f"\n✓ Lag-Llama: {len(ASSETS)} assets saved to {out_dir}")

/home/jovyan/.conda-envs/cfp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


/home/jovyan/.conda-envs/cfp/lib/python3.11/site-packages/lightning/fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Output: cfp_ijf_data/lagllama


Lag-Llama → STOXX:   4%|▍         | 1/24 [2:49:14<64:52:38, 10154.70s/it]

In [ ]:
# Verify outputs
from pathlib import Path
for model_slug, _, label in MODELS:
    out_dir = DATA_DIR / model_slug
    files = list(out_dir.glob('*.parquet'))
    print(f'{label}: {len(files)} parquet files')